In [30]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
import numpy as np
import open3d as o3d
import requests
import dataclasses
from pathlib import Path
import depth_pro
from depth_pro.depth_pro import create_model_and_transforms, DEFAULT_MONODEPTH_CONFIG_DICT

Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']


***Fonctions conversion trimesh--open3d***


In [31]:
import numpy as np
import trimesh
import open3d as o3d


def open3d_to_trimesh(o3d_mesh: o3d.geometry.TriangleMesh) -> trimesh.Trimesh:
    """
    Convert an Open3D TriangleMesh to a Trimesh object.
    """

    if not isinstance(o3d_mesh, o3d.geometry.TriangleMesh):
        raise TypeError("Input must be an Open3D TriangleMesh")

    vertices = np.asarray(o3d_mesh.vertices)
    faces = np.asarray(o3d_mesh.triangles)

    # Normals (optional)
    vertex_normals = None
    if o3d_mesh.has_vertex_normals():
        vertex_normals = np.asarray(o3d_mesh.vertex_normals)

    # Vertex colors (optional)
    vertex_colors = None
    if o3d_mesh.has_vertex_colors():
        vertex_colors = np.asarray(o3d_mesh.vertex_colors)

    mesh = trimesh.Trimesh(
        vertices=vertices,
        faces=faces,
        vertex_normals=vertex_normals,
        vertex_colors=vertex_colors,
        process=False  # avoid automatic cleanup unless desired
    )

    return mesh


In [32]:
def trimesh_to_open3d(tri_mesh: trimesh.Trimesh) -> o3d.geometry.TriangleMesh:
    """
    Convert a Trimesh object to an Open3D TriangleMesh.
    """

    if not isinstance(tri_mesh, trimesh.Trimesh):
        raise TypeError("Input must be a Trimesh object")

    o3d_mesh = o3d.geometry.TriangleMesh()

    # Copie des tableaux pour éviter "array is not writeable" (ex. mesh issu de trimesh.boolean)
    o3d_mesh.vertices = o3d.utility.Vector3dVector(np.asarray(tri_mesh.vertices).copy())
    o3d_mesh.triangles = o3d.utility.Vector3iVector(np.asarray(tri_mesh.faces).copy())

    # Normals
    if tri_mesh.vertex_normals is not None and len(tri_mesh.vertex_normals) == len(tri_mesh.vertices):
        o3d_mesh.vertex_normals = o3d.utility.Vector3dVector(np.asarray(tri_mesh.vertex_normals).copy())
    else:
        o3d_mesh.compute_vertex_normals()

    # Vertex colors (ensure float in [0,1])
    if tri_mesh.visual.kind == 'vertex' and tri_mesh.visual.vertex_colors is not None:
        colors = np.asarray(tri_mesh.visual.vertex_colors[:, :3]).copy()  # drop alpha if present
        if colors.max() > 1.0:
            colors = colors / 255.0
        o3d_mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

    return o3d_mesh


In [33]:
def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)

Génération du modèle deeplearning pour créer une carte de profondeur à partir d'une image RGB

In [34]:
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"


In [ ]:
# Utiliser depth_pro_alt.pt après re-téléchargement (voir cellule markdown au-dessus si erreur "central directory")
CHECKPOINT = Path(r"C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt")
config = dataclasses.replace(DEFAULT_MONODEPTH_CONFIG_DICT, checkpoint_uri=str(CHECKPOINT))
model, transform = create_model_and_transforms(config=config)
model.eval()

image_og, _, f_px = depth_pro.load_rgb(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg")
image = transform(image_og)
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")

In [36]:
prediction = model.infer(image, f_px=f_px)
depth = prediction["depth"].squeeze().cpu().numpy()  # en mètres, numpy pour revert_depth_image()

In [42]:
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
IMAGE_PATH = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image = Image.open(IMAGE_PATH).convert("RGB")
feature_extractor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")
inputs = feature_extractor(images=image, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

Loading weights:   0%|          | 0/972 [00:00<?, ?it/s]

In [38]:
pad = 16
depth_array = predicted_depth.squeeze().cpu().numpy() * 1000.0
depth_array = depth_array[pad:-pad, pad:-pad]
image_cropped = image.crop((pad, pad, image.width - pad, image.height - pad))

In [44]:
# Apercu Procede 2 : image rognee et carte de profondeur GLPN
depth = revert_depth_image(depth)
fig, ax = plt.subplots(1, 2)
ax[0].imshow(image_cropped)
ax[0].set_title("Image rognee")
ax[1].imshow(depth, cmap='plasma')
ax[1].set_title("Profondeur GLPN")
ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout()
plt.show()

In [11]:
image = np.asarray(image)
h2, w2 = len(image[0]), len(image[0][0])
intrinsic_2 = o3d.camera.PinholeCameraIntrinsic(w2, h2, 1000.0, 1000.0, w2 / 2, h2 / 2)

In [12]:
def create_point_cloud_from_depth_image(depth_image, intrinsic, scale=1.0, color_image=None):
    """
    Nuage de points à partir d'une carte de profondeur (numpy 2D).
    Si color_image est fourni (PIL ou numpy H,W,3), les points gardent ces couleurs.
    """
    height, width = depth_image.shape
    depth_o3d = o3d.geometry.Image((depth_image / scale).astype(np.float32))
    if color_image is not None:
        color_np = np.asarray(color_image, dtype=np.uint8)
        if color_np.ndim == 2:
            color_np = np.stack([color_np] * 3, axis=-1)
        if color_np.shape[0] != height or color_np.shape[1] != width:
            pil_img = Image.fromarray(color_np).resize((width, height), Image.Resampling.LANCZOS)
            color_np = np.asarray(pil_img, dtype=np.uint8)
        color_o3d = o3d.geometry.Image(np.ascontiguousarray(color_np))
    else:
        color_o3d = o3d.geometry.Image(np.zeros((height, width, 3), dtype=np.uint8))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=1000.0, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    return o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

In [13]:
# Nuage de points Procédé 2 (carte de profondeur GLPN + couleurs de l'image)
# S'assurer que l'image a le bon format pour color_image (HxWx3, dtype=uint8)
if isinstance(image, torch.Tensor):
    image_np = image.squeeze().permute(1, 2, 0).cpu().numpy()
else:
    image_np = np.array(image)

# Normaliser l'image en uint8 et 3 canaux
if image_np.dtype != np.uint8:
    image_np = (image_np * 255).clip(0, 255).astype(np.uint8)
if image_np.ndim == 2:
    # Image N&B, on la convertit en RGB
    image_np = np.stack([image_np]*3, axis=-1)
if image_np.shape[2] == 1:
    image_np = np.repeat(image_np, 3, axis=2)
elif image_np.shape[2] > 3:
    image_np = image_np[..., :3]


In [14]:
def create_point_cloud_from_image(color_image, depth_image, intrinsic, depth_scale=1000.0, remove_white=True):
    """
    Nuage de points à partir d'une image couleur et d'une carte de profondeur (numpy).
    Utilise Open3D RGBDImage.create_from_color_and_depth + PointCloud.create_from_rgbd_image.
    Si remove_white=True, enlève les points complètement blancs (RGB ≈ 1,1,1).
    """
    h, w = depth_image.shape
    color_o3d = o3d.geometry.Image(np.asarray(color_image, dtype=np.uint8))
    depth_o3d = o3d.geometry.Image(depth_image.astype(np.float32))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_o3d, depth_o3d, depth_scale=depth_scale, depth_trunc=1000.0, convert_rgb_to_intensity=False
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    if remove_white and pcd.has_colors():
        colors = np.asarray(pcd.colors)
        not_white = np.any(colors < 0.65, axis=1)
        indices = np.where(not_white)[0]
        pcd = pcd.select_by_index(indices)
    return pcd

In [15]:
def create_normal_lines(pcd, normal_length=0.0001):
    """
    Crée un LineSet pour visualiser les normales d'un nuage de points.
    Chaque normale est représentée par une ligne partant du point.
    
    Parameters:
    - pcd: PointCloud Open3D avec des normales estimées
    - normal_length: Longueur des lignes de normales à afficher
    
    Returns:
    - LineSet représentant les normales
    """
    if not pcd.has_normals():
        return None
    
    pts = np.asarray(pcd.points)
    normals = np.asarray(pcd.normals)
    n = len(pts)
    
    # Créer les points de départ et d'arrivée pour chaque normale
    pts_start = pts
    pts_end = pts + normals * normal_length
    pts_all = np.vstack([pts_start, pts_end])
    
    # Créer les lignes : chaque ligne relie le point i au point i+n
    lines = np.array([[i, i + n] for i in range(n)], dtype=np.int32)
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(pts_all)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    # Couleur bleue pour les normales
    line_set.colors = o3d.utility.Vector3dVector(np.tile([[0, 0, 1]], (n, 1)))
    
    return line_set
points = create_point_cloud_from_depth_image(depth_array, intrinsic_2, scale=1.0, color_image=image_cropped)




In [16]:

pcd_glpn = create_point_cloud_from_depth_image(depth, intrinsic_2, scale=1.0, color_image=image_np)
pcd_glpn.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)

# Création des lignes de normales
normal_lines = create_normal_lines(pcd_glpn, normal_length=0.000005)
o3d.visualization.draw_geometries([pcd_glpn], window_name="Procede 2 - GLPN")

In [17]:
def point_cloud_to_closed_mesh_alpha_shape(pcd, alpha, clean=True):
    """
    Convertit un nuage de points (Open3D) en maillage fermé par alpha shape (alpha wrapping).
    alpha : plus petit = surface plus détaillée, plus grand = surface plus lisse.
    clean : si True, supprime triangles dégénérés/dupliqués et recalcule les normales.
    """
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    if clean:
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    return mesh

In [18]:
# Normales du nuage : estimation puis orientation cohérente (vers caméra au-dessus pour une pile vue du dessus)
if not pcd_glpn.has_normals():
    pcd_glpn.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
    )
center = np.asarray(pcd_glpn.points).mean(axis=0)
camera_above = center + np.array([0, 0, 1.0])  # au-dessus du nuage pour orienter les normales vers l'extérieur
pcd_glpn.orient_normals_towards_camera_location(camera_above)




ancienne méthode alpha shape (delaunay)

In [19]:
# tetra_mesh, pt_map = o3d.geometry.TetraMesh.create_from_point_cloud(pcd_glpn)
# alpha = 0.000045
# print(f"alpha={alpha:.6f}")
# mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
#         pcd_glpn, alpha, tetra_mesh, pt_map)

# # Ajuster les normales du mesh : orientation cohérente puis vers l'extérieur
# mesh.compute_vertex_normals()
# # La méthode 'orient_triangles_consistent()' n'existe pas dans Open3D.
# # À la place, réordonner les triangles pour normaliser leur orientation (déjà fait ci-dessous)
# # (aucune action ici, le code suivant gère manuellement l'orientation)
# mesh_center = mesh.get_center()
# verts = np.asarray(mesh.vertices)
# tris = np.asarray(mesh.triangles)
# for i in range(len(tris)):
#     a, b, c = verts[tris[i]]
#     n = np.cross(b - a, c - a)
#     n /= (np.linalg.norm(n) + 1e-10)
#     centroid = (a + b + c) / 3
#     if np.dot(n, centroid - mesh_center) < 0:
#         tris[i] = tris[i][[0, 2, 1]]
# mesh.triangles = o3d.utility.Vector3iVector(tris)
# mesh.compute_vertex_normals()
#mesh = open3d_to_trimesh(mesh)

In [20]:
def display_mesh(mesh, window_name="Mesh"):
    """
    Affiche un mesh dans le notebook (Open3D ou trimesh.Trimesh).
    En trimesh, conversion vers Open3D puis visualisation WebRTC.
    """
    if isinstance(mesh, trimesh.Trimesh):
        # Conversion trimesh -> Open3D (copie des tableaux pour éviter "array is not writeable")
        o3d_mesh = o3d.geometry.TriangleMesh()
        o3d_mesh.vertices = o3d.utility.Vector3dVector(np.asarray(mesh.vertices).copy())
        o3d_mesh.triangles = o3d.utility.Vector3iVector(np.asarray(mesh.faces).copy())
        if mesh.vertex_normals is not None and len(mesh.vertex_normals) == len(mesh.vertices):
            o3d_mesh.vertex_normals = o3d.utility.Vector3dVector(np.asarray(mesh.vertex_normals).copy())
        else:
            o3d_mesh.compute_vertex_normals()
        if hasattr(mesh.visual, "vertex_colors") and mesh.visual.vertex_colors is not None:
            colors = np.asarray(mesh.visual.vertex_colors[:, :3]).copy()
            if colors.max() > 1.0:
                colors = colors / 255.0
            o3d_mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
        mesh = o3d_mesh
    o3d.visualization.draw_geometries([mesh], window_name=window_name)


## Comparaison des algorithmes de triangulation pour `pcd_glpn`

**Contexte** : `pcd_glpn` provient d'une carte de profondeur GLPN (vue du dessus d'une pile de terre). Nuage dense (~500k–900k pts), structuré (grille RGBD), bruit typique des modèles monocular.

| Algorithme | Principe | Vitesse | Qualité surface | Trous | Bruit | Pertinence pcd_glpn |
|------------|----------|---------|-----------------|-------|-------|---------------------|
| **Poisson** | Implicite (octree + FEM) | Rapide (~20 s) | Fermée, lisse | Non | Lisse bien | ⭐⭐⭐ **Idéal** |
| **Alpha Shape** | Convexe → creusage par sphère α | Lent (TetraMesh) | Détaillée mais fragile | Peu | Sensible | ⭐⭐ Moyen |
| **Ball Pivoting** | Sphère pivotante locale | Très lent (O(n²)) | Topologie exacte | Oui | Sensible | ⭐ Faible |

---

### Poisson (recommandé)
- **Avantages** : Surface fermée, lisse, rapide, gère bien le bruit des cartes de profondeur.
- **Limites** : Peut over-smoother ; extrapole dans les zones peu denses (filtrer via `density_quantile`).

### Alpha Shape (déjà utilisé en cellule 16)
- **Avantages** : Suit les points de près, pas d’extrapolation.
- **Limites** : Les warnings `invalid tetra in TetraMesh` montrent une fragilité sur données denses/proches du planaire ; `alpha=0.000045` très fin, réglage délicat ; TetraMesh coûteux en mémoire et temps pour ~900k points.

### Ball Pivoting
- **Avantages** : Topologie fidèle, sommets = points d’entrée.
- **Limites** : Très lent, laisse des trous, sensible au bruit et à la densité.

→ Pour `pcd_glpn`, **Poisson reste le meilleur choix**. Ball Pivoting optimisé (voxel down) utile pour comparaison uniquement.

In [21]:
# --- Option 1 : Poisson (recommandé pour pcd_glpn) - rapide et surface fermée ---
def mesh_poisson(pcd, depth=8, density_quantile=0.03):
    """Reconstruction Poisson : surface lisse et fermée, ~20x plus rapide que ball pivoting."""
    if not pcd.has_normals():
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=depth)
    densities = np.asarray(densities)
    mesh = mesh.select_by_index(np.where(densities > np.quantile(densities, density_quantile))[0])
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    return mesh

rec_mesh = mesh_poisson(pcd_glpn, depth=8, density_quantile=0.03)
print("Poisson:", rec_mesh)
o3d.visualization.draw_geometries([pcd_glpn, rec_mesh], window_name="Poisson (recommandé)")




Poisson: TriangleMesh with 76186 points and 151684 triangles.


Bord du mesh et fermeture (manifold)
- Extraction des boucles de bord sans HalfEdgeTriangleMesh (evite l'erreur "Duplicated half-edges" sur le mesh Poisson).
- Visualisation du bord en rouge.

In [47]:
rec_mesh2 = mesh_poisson(pcd_glpn, depth=5, density_quantile = 0.2)
o3d.visualization.draw_geometries([rec_mesh, rec_mesh2], window_name="Poisson 2 (recommandé)")


In [23]:
import open3d as o3d
import numpy as np
from collections import defaultdict

def show_mesh_boundaries(mesh, window_name="Boundaries Visualization", verbose=True):
    """
    Trouve et affiche les boucles de bord d'un mesh Open3D.
    Renvoie la liste des boucles de bord (liste d'indices).
    Affiche le mesh et les boucles en rouge.

    Args:
        mesh (o3d.geometry.TriangleMesh): Mesh à analyser.
        window_name (str): Nom de la fenêtre d'affichage.
        verbose (bool): Affiche le nombre de boucles.
    Returns:
        list[list[int]]: Liste des boucles, chaque boucle est une liste d'indices.
    """
    def get_boundary_loops(mesh):
        """Boucles de bords (aretes dans un seul triangle). Ne modifie pas le mesh."""
        tris = np.asarray(mesh.triangles)
        edge_count = defaultdict(int)
        for t in tris:
            a, b, c = int(t[0]), int(t[1]), int(t[2])
            if a == b or b == c or c == a:
                continue
            for u, v in [(a, b), (b, c), (c, a)]:
                edge_count[(min(u, v), max(u, v))] += 1
        boundary_edges = [(u, v) for (u, v), c in edge_count.items() if c == 1]
        if not boundary_edges:
            return []
        adj = defaultdict(list)
        for u, v in boundary_edges:
            adj[u].append(v)
            adj[v].append(u)
        seen_edges = set()
        loops = []
        for u, v in boundary_edges:
            edge_key = (min(u, v), max(u, v))
            if edge_key in seen_edges:
                continue
            loop = [u]
            cur, prev = u, v
            while True:
                seen_edges.add((min(cur, prev), max(cur, prev)))
                next_cands = [x for x in adj[cur] if x != prev]
                if not next_cands:
                    break
                nxt = next_cands[0]
                loop.append(nxt)
                if nxt == u:
                    break
                prev, cur = cur, nxt
            if len(loop) > 2:
                loops.append(loop)
        return loops

    boundaries = get_boundary_loops(mesh)
    boundary_geom_list = [mesh]
    for b in boundaries:
        if len(b) > 1:
            points = np.asarray(mesh.vertices)[b]
            lines = [[i, (i + 1) % len(points)] for i in range(len(points))]
            color = [[1, 0, 0] for _ in range(len(lines))]
            line_set = o3d.geometry.LineSet()
            line_set.points = o3d.utility.Vector3dVector(points)
            line_set.lines = o3d.utility.Vector2iVector(lines)
            line_set.colors = o3d.utility.Vector3dVector(color)
            boundary_geom_list.append(line_set)
    if verbose:
        print(f"Nombre de boucles de bord : {len(boundaries)}")
    o3d.visualization.draw_geometries(boundary_geom_list, window_name=window_name)
    return boundaries

# Exemple d'appel (remplacez rec_mesh par n'importe quel mesh Open3D) :
# boundaries = show_mesh_boundaries(rec_mesh, window_name="Boundaries Visualization")

Extrusion du bord vers le bas (base horizontale plate)
- Duplication de la boucle de bord a une Z inferieure (paroi verticale).
- Fond plat a la meme Z pour toute la boucle (geometrie horizontalement plate).

In [45]:
def extrude_boundary_down(mesh, boundary, offset_ratio=0.1, visualize=True, window_name="rec_mesh avec extrusion"):
    """
    Effectue une extrusion vers le bas d'une boucle de bord 'boundary' du mesh fourni.
    - Duplique la boucle à un Z inférieur (paroi verticale).
    - Ajoute un fond plat à la même Z (géométrie horizontalement plate).

    Args:
        mesh (o3d.geometry.TriangleMesh): Mesh d'entrée.
        boundary (list ou np.ndarray): Indices des sommets (ordre) formant la boucle de bord à extruder.
        offset_ratio (float): Pourcentage de la hauteur totale utilisé comme profondeur de l'extrusion si z_bottom non spécifié.
        visualize (bool): Affiche le mesh résultant si True.
        window_name (str): Titre de la fenêtre de visualisation.

    Returns:
        o3d.geometry.TriangleMesh: Nouveau mesh après extrusion et ajout du fond plat.
    """
    import numpy as np
    import open3d as o3d

    b = list(boundary)
    if len(b) < 3:
        raise ValueError("Boucle de bord trop courte.")

    verts = np.asarray(mesh.vertices).copy()
    tris = np.asarray(mesh.triangles).copy()
    n_verts = len(verts)
    pts_b = verts[b]
    z_top = float(np.mean(pts_b[:, 2]))
    z_bottom = float(np.min(pts_b[:, 2])) - max(1e-6, (np.ptp(verts[:, 2]) * offset_ratio))  # 10% de la hauteur ou petit offset
    extrusion_depth = z_top - z_bottom

    # Sommets du bas (même X,Y, Z = z_bottom)
    bottom_verts = pts_b.copy()
    bottom_verts[:, 2] = z_bottom
    new_bottom_indices = np.arange(n_verts, n_verts + len(b))
    verts = np.vstack([verts, bottom_verts])
    # Centre du fond (pour la cap)
    center_bottom = np.mean(bottom_verts, axis=0)
    center_idx = len(verts)
    verts = np.vstack([verts, center_bottom.reshape(1, 3)])

    # Paroi verticale : quads (b[i], b[i+1], new_b[i+1], new_b[i]) en 2 triangles
    wall_tris = []
    for i in range(len(b)):
        i1 = (i + 1) % len(b)
        vi, vi1 = b[i], b[i1]
        ni, ni1 = new_bottom_indices[i], new_bottom_indices[i1]
        wall_tris.append([vi, vi1, ni1])
        wall_tris.append([vi, ni1, ni])
    # Fond plat : eventail depuis le centre
    cap_tris = [[center_idx, new_bottom_indices[i], new_bottom_indices[(i + 1) % len(b)]] for i in range(len(b))]

    extruded_mesh = o3d.geometry.TriangleMesh(
        o3d.utility.Vector3dVector(verts),
        o3d.utility.Vector3iVector(np.vstack([tris, np.array(wall_tris), np.array(cap_tris)]))
    )
    extruded_mesh.compute_vertex_normals()
    print(f"Extrusion vers le bas : Z de {z_top:.4f} a {z_bottom:.4f} (profondeur {extrusion_depth:.4f}). Fond plat ajouté.")
    if visualize:
        o3d.visualization.draw_geometries([extruded_mesh], window_name=window_name)
    return extruded_mesh



In [46]:
boundaries1 = show_mesh_boundaries(rec_mesh,"name1", False)
bounded_rec_mesh = extrude_boundary_down(rec_mesh, boundaries1[0],0.1, True,"name")
boundaries2 = show_mesh_boundaries(rec_mesh2,"name1", False)
bounded_rec_mesh2 = extrude_boundary_down(rec_mesh2, boundaries2[0],0.1, True,"name")

Extrusion vers le bas : Z de 0.0015 a 0.0014 (profondeur 0.0002). Fond plat ajouté.
Extrusion vers le bas : Z de 0.0015 a 0.0014 (profondeur 0.0001). Fond plat ajouté.


Geometrie edge-manifold a coup sur
Pipeline Open3D : nettoyage puis suppression des aretes non manifold. Le mesh resultant est garanti edge-manifold (et en pratique vertex-manifold).

In [26]:
rec_mesh.is_vertex_manifold()
rec_mesh2.is_vertex_manifold()

True

Difference de volume (booléen) : rec_mesh - rec_mesh2
- Les deux meshes sont convertis en format tensor Open3D pour `boolean_difference()`.
- Resultat : volume de rec_mesh prive de l'intersection avec rec_mesh2. Les deux meshes doivent etre fermes et edge-manifold pour un resultat fiable.

In [27]:
# Difference de volume : rec_mesh - rec_mesh2 (soustraction booléenne)
# Open3D tensor (boolean_difference) provoque des crashs kernel sur meshes Poisson ; on utilise trimesh à la place.
tm_a = open3d_to_trimesh(rec_mesh)
tm_b = open3d_to_trimesh(rec_mesh2)
# check_volume=False pour éviter une erreur si les meshes ne sont pas parfaitement watertight
rec_mesh_diff = trimesh_to_open3d(trimesh.boolean.difference([tm_a, tm_b], engine=None, check_volume=False))
rec_mesh_diff.compute_vertex_normals()
print(f"Difference (rec_mesh - rec_mesh2) : {len(rec_mesh_diff.vertices)} sommets, {len(rec_mesh_diff.triangles)} triangles")
o3d.visualization.draw_geometries([rec_mesh_diff], window_name="rec_mesh_diff (difference de volume)")

Difference (rec_mesh - rec_mesh2) : 0 sommets, 0 triangles
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.
